In [1]:
import os

In [2]:
from openai import AzureOpenAI

In [3]:
from azure.search.documents import SearchClient

In [4]:
from azure.core.credentials import AzureKeyCredential

In [5]:
from azure.search.documents.models import VectorizedQuery, VectorizableTextQuery

In [6]:
from dotenv import load_dotenv, find_dotenv

In [7]:
# Busca el .env en esta carpeta o en las carpetas superiores (ej. C:\Users\alexc\Proyectos)

In [8]:
load_dotenv(find_dotenv(usecwd=True), override=True)

True

In [9]:
OPENAI_API_TYPE = "azure"
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")

AZURE_OPENAI_EMBEDDING_DEPLOYED_MODEL_NAME = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")  # text-embedding-3-small
AZURE_OPENAI_CHAT_DEPLOYED_MODEL_NAME = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")            # gpt-4.1-mini

AZURE_SEARCH_SERVICE_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT")
AZURE_SEARCH_INDEX_NAME = "rag-1791093456903"
AZURE_SEARCH_ADMIN_KEY = os.getenv("AZURE_SEARCH_KEY")

In [10]:
# Verificación rápida (no imprime las claves)

In [11]:
for nombre, valor in {
    "AZURE_OPENAI_API_KEY": AZURE_OPENAI_API_KEY,
    "AZURE_OPENAI_ENDPOINT": AZURE_OPENAI_ENDPOINT,
    "EMBEDDING": AZURE_OPENAI_EMBEDDING_DEPLOYED_MODEL_NAME,
    "CHAT": AZURE_OPENAI_CHAT_DEPLOYED_MODEL_NAME,
    "AZURE_SEARCH_SERVICE_ENDPOINT": AZURE_SEARCH_SERVICE_ENDPOINT,
    "AZURE_SEARCH_ADMIN_KEY": AZURE_SEARCH_ADMIN_KEY,
}.items():
    print(f"{nombre}: {'✅' if valor else '❌ FALTA'}")

AZURE_OPENAI_API_KEY: ✅
AZURE_OPENAI_ENDPOINT: ✅
EMBEDDING: ✅
CHAT: ✅
AZURE_SEARCH_SERVICE_ENDPOINT: ✅
AZURE_SEARCH_ADMIN_KEY: ✅


In [12]:
openai_client = AzureOpenAI(
    api_key = AZURE_OPENAI_API_KEY,
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    api_version= OPENAI_API_VERSION
)

In [13]:
search_client = SearchClient(
    endpoint = AZURE_SEARCH_SERVICE_ENDPOINT,
    index_name = AZURE_SEARCH_INDEX_NAME,
    credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
)

In [14]:
def get_embedding(text):
    return openai_client.embeddings.create(
        model=AZURE_OPENAI_EMBEDDING_DEPLOYED_MODEL_NAME,
        input= text
    ).data[0].embedding

In [15]:
user_question = "What is included in my Noethwind Health plus plan that is not in stardard"
user_question_vector = get_embedding(user_question)
print(user_question_vector)

[-0.0014886856079101562, 0.0277557373046875, 0.032012939453125, 0.03448486328125, -0.0435791015625, 0.0082244873046875, -0.049652099609375, 0.018035888671875, 0.010345458984375, 0.0105743408203125, 0.020263671875, 0.00962066650390625, 0.00421142578125, 0.0139617919921875, -0.004833221435546875, 0.036956787109375, -0.03045654296875, -0.0021076202392578125, -0.0002605915069580078, 0.05377197265625, 0.018585205078125, 0.04705810546875, 0.006328582763671875, -0.035675048828125, -0.039337158203125, -0.0159454345703125, -0.025787353515625, -0.01094818115234375, 0.049560546875, 0.0008893013000488281, 0.0034542083740234375, -0.0390625, 0.01336669921875, 0.007656097412109375, -0.009796142578125, -0.01235198974609375, 0.02777099609375, 0.0184326171875, 0.010711669921875, 0.022674560546875, -0.02288818359375, 0.01386260986328125, 0.0180206298828125, 0.0235443115234375, 0.01012420654296875, 0.0020236968994140625, 0.0543212890625, -0.008941650390625, -0.0281219482421875, 0.032958984375, -0.02995300

In [20]:
search_results = search_client.search(
    None,
    top = 3,
    vector_queries=[
        VectorizableTextQuery(
            text=user_question, k_nearest_neighbors=3, fields="text_vector"
        )
    ],
)

In [21]:
for result in search_results:
    print("Chunk ID:", result["chunk_id"])
    print ("Title:", result["title"])    
    print ("Text:", result["chunk"])
    print

Chunk ID: 1de8ea84a099_aHR0cHM6Ly9zdG9yYWdlcmFncGxhdHppMjU3OTcuYmxvYi5jb3JlLndpbmRvd3MubmV0L3JhZy1kb2N1bWVudHMvQmVuZWZpdF9PcHRpb25zLnBkZg2_pages_1
Title: Benefit_Options.pdf
Text: a variety of in-network providers, including primary care 
physicians, specialists, hospitals, and pharmacies. This plan does not offer coverage for emergency 
services, mental health and substance abuse coverage, or out-of-network services.

Comparison of Plans 
Both plans offer coverage for routine physicals, well-child visits, immunizations, and other preventive 
care services. The plans also cover preventive care services such as mammograms, colonoscopies, and 
other cancer screenings. 

Northwind Health Plus offers more comprehensive coverage than Northwind Standard. This plan offers 
coverage for emergency services, both in-network and out-of-network, as well as mental health and 
substance abuse coverage. Northwind Standard does not offer coverage for emergency services, mental 
health and substance ab

In [27]:
AZURE_OPENAI_CHAT_COMPLETION_DEPLOYED_MODEL_NAME = "gpt-4.1-mini"

In [28]:
context = ""
for result in search_results:
    context += result["chunk"] + "\n\n"

SYSTEM_MESSAGE = f"""
You are an AI Assistant.
Be brief in your answers. Answer ONLY with the facts listed in the retrieved text.

Context: 
{context}
"""

In [29]:
USER_MESSAGE = user_question

In [30]:
response = openai_client.chat.completions.create(
    model=AZURE_OPENAI_CHAT_COMPLETION_DEPLOYED_MODEL_NAME,
    temperature = 0.7,
    messages=[
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": USER_MESSAGE},
    ],
)

In [31]:
answer = response.choices[0].message.content

In [32]:
print(answer)

The Noethwind Health plus plan includes coverage for dental care, optical care, and wellness programs, which are not included in the standard plan.
